# Qwen3-0.6B Conversational Fine-Tuning with LoRA (Colab)

This notebook fine-tunes `Qwen/Qwen3-0.6B` using supervised fine-tuning (SFT) and a LoRA adapter.

**Target environment:** Google Colab with a CUDA GPU (for example, NVIDIA T4).

## Important
- This notebook uses standard LoRA, not QLoRA or AMD ExecuTorch quantization-aware training.
- It checks the GPU and BF16 capability automatically.
- It pins a compatible software stack to reduce dependency drift.
- The sample dataset is for experimentation. Review the dataset license and suitability before using the resulting model.
- Run cells from top to bottom. After installing packages, restart the runtime if Colab asks you to; then start again from the GPU check.

## 1. Check the Colab GPU

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. In Colab choose Runtime > Change runtime type > GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
use_bf16 = torch.cuda.is_bf16_supported()

print("GPU:", gpu_name)
print("CUDA:", torch.version.cuda)
print("BF16 supported:", use_bf16)
print("Training dtype:", "bfloat16" if use_bf16 else "float16")


GPU: Tesla T4
CUDA: 13.0
BF16 supported: True
Training dtype: bfloat16


## 2. Install a consistent training stack

This pins a known compatible-style stack instead of upgrading every package to the newest release. If these versions conflict with the PyTorch build preinstalled in your Colab runtime, use a fresh Colab runtime and rerun this cell.

In [2]:
%pip install -q "transformers==4.51.3" "trl==0.18.2" "peft==0.15.2" "datasets>=3.0.0,<4.0.0" "accelerate>=1.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is i

## 3. Verify package versions and GPU after installation

In [3]:
import torch, transformers, trl, peft, datasets, accelerate
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime and restart.")


PyTorch: 2.11.0+cu130
Transformers: 4.51.3
TRL: 0.18.2
PEFT: 0.15.2
Datasets: 3.6.0
Accelerate: 1.15.0
CUDA available: True


## 4. Load Qwen3-0.6B

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
)
model.config.use_cache = False

print("Loaded model:", MODEL_NAME)
print("Tokenizer has chat template:", tokenizer.chat_template is not None)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-0.6B
Tokenizer has chat template: True


## 5. Load and inspect the conversational dataset

In [5]:
from datasets import load_dataset

raw_dataset = load_dataset("trl-lib/Capybara", split="train")
print("Columns:", raw_dataset.column_names)
print("First row:", raw_dataset[0])
print("Rows:", len(raw_dataset))


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/37.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/482k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Columns: ['source', 'messages', 'num_turns']
First row: {'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of An

## 6. Normalize dataset into TRL conversational format

The output format is a list of `{role, content}` messages. If the upstream dataset schema changes, inspect the previous cell's sample and adapt the normalization function.

In [6]:
def normalize_conversation(example):
    # Most conversational datasets already provide messages.
    msgs = example.get("messages")
    if isinstance(msgs, list):
        normalized = []
        for msg in msgs:
            if not isinstance(msg, dict):
                continue
            role = msg.get("role")
            content = msg.get("content")
            if role in {"system", "user", "assistant"} and isinstance(content, str):
                normalized.append({"role": role, "content": content})
        if any(m["role"] == "user" for m in normalized) and any(m["role"] == "assistant" for m in normalized):
            return {"messages": normalized}

    # Prompt/completion or instruction/output formats.
    prompt = example.get("prompt") or example.get("instruction") or example.get("question")
    answer = example.get("completion") or example.get("output") or example.get("response") or example.get("answer")
    if isinstance(prompt, str) and isinstance(answer, str):
        return {"messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer},
        ]}

    # Common conversations format.
    conv = example.get("conversations")
    if isinstance(conv, list):
        normalized = []
        for turn in conv:
            if not isinstance(turn, dict):
                continue
            role = turn.get("role")
            content = turn.get("content", turn.get("value"))
            role = {"human": "user", "gpt": "assistant"}.get(role, role)
            if role in {"system", "user", "assistant"} and isinstance(content, str):
                normalized.append({"role": role, "content": content})
        if any(m["role"] == "user" for m in normalized) and any(m["role"] == "assistant" for m in normalized):
            return {"messages": normalized}

    return {"messages": []}

dataset = raw_dataset.map(
    normalize_conversation,
    remove_columns=raw_dataset.column_names,
)
dataset = dataset.filter(
    lambda row: len(row["messages"]) >= 2
    and any(m["role"] == "user" for m in row["messages"])
    and any(m["role"] == "assistant" for m in row["messages"])
)

if len(dataset) < 2:
    raise ValueError(
        "Could not normalize enough rows. Inspect the raw dataset schema and update normalize_conversation()."
    )

print("Usable rows:", len(dataset))
print("Normalized sample:", dataset[0])


Map:   0%|          | 0/15806 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15806 [00:00<?, ? examples/s]

Usable rows: 15806
Normalized sample: {'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfully

## 7. Make train and validation splits

In [7]:
splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print("Train examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))


Train examples: 15015
Validation examples: 791


## 8. Configure LoRA

In [8]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

print(lora_config)


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'v_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)


## 9. Configure SFT and train

This configuration deliberately uses a standard negative log-likelihood loss and disables Liger kernels. The `loss_type` argument is version-dependent; the pinned TRL version should support the configuration below. The notebook avoids relying on a chunked BF16 log-probability path.

In [12]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "./qwen3-0.6b-lora"
MAX_STEPS = 100  # Set to -1 for one full epoch; 100 is a shorter starter run.

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    max_length=1024,
    fp16=not use_bf16,
    bf16=use_bf16,

    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to="none",
    seed=42,


    use_liger_kernel=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

train_result = trainer.train()
print(train_result)


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss
50,1.382900,1.541198
100,1.588900,1.526071


TrainOutput(global_step=100, training_loss=1.5725128364562988, metrics={'train_runtime': 2119.9313, 'train_samples_per_second': 0.377, 'train_steps_per_second': 0.047, 'total_flos': 1547103191040000.0, 'train_loss': 1.5725128364562988})


## 10. Save the LoRA adapter

In [13]:
ADAPTER_PATH = "./qwen3-0.6b-lora-final"

trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved to:", ADAPTER_PATH)


Adapter saved to: ./qwen3-0.6b-lora-final


## 11. Test the fine-tuned assistant

In [3]:
import torch

model.eval()
model.config.use_cache = True

messages = [
    {
        "role": "system",
        "content": "You are a helpful, clear, and friendly assistant."
    },
    {
        "role": "user",
        "content": "Explain RAG in simple terms and give one example."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))

**RAG** stands for **Retrieval-Augmented Generation**, which is a method that helps you generate text or other content by first retrieving relevant information from a database or knowledge base. It's like using a library of information to help you create new content more efficiently.

**Example:**  
If you're writing a blog post about the importance of exercise, RAG can help you find articles or studies that talk about the benefits of exercise and then use that information to write a well-structured article.


## 12. Optional: save adapter to Google Drive

Run this if you want the adapter to survive a Colab runtime reset.

In [16]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
drive_path = "/content/drive/MyDrive/qwen3-0.6b-lora-final"
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)
shutil.copytree(ADAPTER_PATH, drive_path)
print("Copied adapter to:", drive_path)


Mounted at /content/drive
Copied adapter to: /content/drive/MyDrive/qwen3-0.6b-lora-final


## Troubleshooting

- **GPU not found:** Select a GPU runtime in Colab.
- **CUDA out of memory:** Set `max_length=512` and keep batch size at 1.
- **BF16 kernel error:** The notebook chooses BF16 only when the active GPU reports support; otherwise it uses FP16.
- **Package/API error:** Use a fresh runtime, run the installation cell, restart if requested, and execute cells from the top. Avoid mixing old package installs in the same runtime.
- **Dataset schema changed:** Inspect `raw_dataset[0]` and update the normalizer.
- **Evaluation:** This notebook uses a validation split, but you should also test the base and fine-tuned models on a small set of held-out prompts. A lower loss alone does not guarantee better assistant behavior.